In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import random
import json
import numpy as np
import math
from PIL import Image, ImageDraw
import re
import ast


df1 = pd.read_json(f'/path/to/avi-math/label.json')
df2 = pd.read_csv(f'/path/to/avi-math/gpt4o-extract-ans.csv', names=['qid', 'response'])
df = pd.merge(df1, df2, on='qid', how='inner')

def compute_acc(filtered_df, print_flag=True):
    correct_predictions = 0    
    total = len(filtered_df)

    for index, row in filtered_df.iterrows():
        answer = row['answer']
        response = row['response']
        
        if row['eva'] == 'str':
            ANSWER_PATTERN_MULTICHOICE = r"(?i)Answer\s*:\s*([A-Za-z]+)"
        else:
            ANSWER_PATTERN_MULTICHOICE = r"(?i)Answer\s*:\s*(.*)"
        
        match = re.search(ANSWER_PATTERN_MULTICHOICE, response)
        if match: 
            prediction = match.group(1)
        else:
            prediction = response
        
        prediction = prediction.replace('meters', '').replace('RMB', '').replace('seconds', '').replace('degrees', '').replace('square meters', '').strip()
    
        if row['eva'] == 'list':
            try:
                prediction = ast.literal_eval(prediction)
                answer = ast.literal_eval(answer)
                i = 0
                for pre, ans in zip(prediction,  answer):
                    if int(pre) == int(ans):
                        i += 1
            except:
                continue

            if i != len(answer):
                continue
        
        else:
            prediction = prediction.replace('$', '').replace(',', '').replace('"', '').strip()
            
            try:
                if row['eva'] == 'int':
                    answer = int(answer)
                    prediction = int(prediction)
                elif row['eva'] == 'str':
                    answer = answer.lower()
                    prediction = prediction.lower()
                elif row['eva'] == 'option':
                    answer = answer.upper()
                    prediction = prediction.upper()
                elif row['eva'] == 'float':
                    answer = round(float(answer), 1)
                    prediction = round(float(prediction) , 1)      
            except:
                continue

            if answer == prediction:
                correct_predictions += 1
                continue
        
        if row['eva'] == 'str' and answer in ['blue', 'red']:     
            filtered_option = [x for x in ['blue', 'red'] if x != answer]
            if answer in prediction and filtered_option[0] not in prediction:
                correct_predictions += 1   

                
    accuracy = correct_predictions / total
    if print_flag:
        print(round(accuracy, 4)) 
    return accuracy

In [7]:
score_list_list = []
for domain in sorted(df['subject'].unique()):
    print(f'============{domain}============')
    df1 = df[df['subject'] == domain]
    total = len(df1)
    total_acc = 0
    total_acc2 = 0
    i = 0
    score_list = []
    for category in sorted(df1['topic'].unique()):
        print(category)
#         print(len(filtered_df))
        filtered_df = df1[df1['topic'] == category]
        if len(filtered_df) == 0:
            continue
        score = compute_acc(filtered_df, print_flag=False)
        score_list.append(score)
        total_acc += score * len(filtered_df)
        total_acc2 += score
        i += 1
    score_list_list.append(score_list)
    weighted_avg_acc = total_acc / total
    avg_acc = total_acc2 / i
score_csv = pd.DataFrame(score_list_list)
score_csv.to_clipboard(index=False, header=False)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 100)
score_csv

============algebra============
multivariate algebra
univariate algebra
============arithmetic============
addition
integer division
multiplication
subtraction
============counting============
counting based on comparison
counting based on multiple properties
counting based on single property
============geometry============
metric geometry
perspective geometry
spatial relationship
============logic============
comparison
deductive reasoning
inductive reasoning
============statistics============
maximum
mean
median
minimum
mode


,0,1,2,3,4
0,0.266667,0.447699,NaN,NaN,NaN
1,0.321244,0.266667,0.117647,0.260504,NaN
2,0.541667,0.183333,0.283333,NaN,NaN
3,0.222917,0.163889,0.079167,NaN,NaN
4,0.337979,0.541667,0.566667,NaN,NaN
5,0.475000,0.250000,0.275000,0.516667,0.664773


In [8]:
score_csv.mean(axis=1)

0    0.357183
1    0.241515
2    0.336111
3    0.155324
4    0.482104
5    0.436288
dtype: float64

In [9]:
score_csv.mean(axis=1).mean()

0.33475421246384385

In [10]:
df1 = df[df['AGL'] .isin([20, 30, 40])]
print(f'============Low Length: {len(df1)}============')
compute_acc(df1)


df1 = df[df['AGL'] .isin([60, 70, 80])]
print(f'============Med Length: {len(df1)}============')
compute_acc(df1)


df1 = df[df['AGL'] .isin([100,110, 120])]
print(f'============High Length: {len(df1)}============')
compute_acc(df1)

for step in sorted(df['pitch_angle'].unique()):
    df1 = df[df['pitch_angle'] == step]
    print(f'============{step} Length: {len(df1)}============')
    compute_acc(df1)

============Low Length: 1428============
0.3676
============Med Length: 1143============
0.3167
============High Length: 1202============
0.2987
============45 Length: 1230============
0.3073
============60 Length: 1324============
0.3406
============90 Length: 1219============
0.3421


In [11]:
df1 = df[df['step'] .isin([2, 3])]
print(f'============Short Length: {len(df1)}============')
compute_acc(df1)


df1 = df[df['step'] .isin([4, 5])]
print(f'============Med Length: {len(df1)}============')
compute_acc(df1)


df1 = df[df['step'] .isin([6])]
print(f'============Long Length: {len(df1)}============')
compute_acc(df1)

============Short Length: 2555============
0.3824
============Med Length: 858============
0.2855
============Long Length: 360============
0.0667


0.06666666666666667

In [12]:
for step in sorted(df['qtype'].unique()):
    df1 = df[df['qtype'] == step]
    print(f'============{step} Length: {len(df1)}============')
    compute_acc(df1)

============free-form Length: 2181============
0.1875
============multiple-choice Length: 1352============
0.5081
============yes/no Length: 240============
0.625


In [13]:
for step in sorted(df['subject'].unique()):
    df1 = df[df['subject'] == step]
    print(f'============{step} Length: {len(df1)}============')
    compute_acc(df1)

============algebra Length: 359============
0.3872
============arithmetic Length: 551============
0.2523
============counting Length: 360============
0.3361
============geometry Length: 1080============
0.1713
============logic Length: 767============
0.4733
============statistics Length: 656============
0.4558
